In [34]:
import pandas as pd
import numpy as np

In [35]:
df = pd.read_csv(r"C:\Users\MERT\Downloads\archive (2)\online_retail_II.csv")

In [36]:
df.shape

(1067371, 8)

In [37]:
df.dtypes

Invoice            str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
Price          float64
Customer ID    float64
Country            str
dtype: object

In [38]:
df.isna()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...
1067366,False,False,False,False,False,False,False,False
1067367,False,False,False,False,False,False,False,False
1067368,False,False,False,False,False,False,False,False
1067369,False,False,False,False,False,False,False,False


In [39]:
df.isna().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

In [40]:
df.duplicated()

0          False
1          False
2          False
3          False
4          False
           ...  
1067366    False
1067367    False
1067368    False
1067369    False
1067370    False
Length: 1067371, dtype: bool

In [41]:
df.duplicated().sum()

np.int64(34335)

In [42]:
df["Invoice"] = df["Invoice"].astype(str)
cancelled = df["Invoice"].str.startswith(("c", "C"), na=False)


In [43]:
df_cancelled = df[cancelled].copy()

In [44]:
df = df[~cancelled].copy()

In [45]:
df.columns = df.columns.str.strip()

In [46]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

In [47]:
df = df.dropna(subset=["Customer ID"])

In [48]:
len(df)

805620

In [49]:
df = df.drop_duplicates()

In [50]:
len(df)

779495

In [51]:
df.head(5)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [52]:
df["total_amount"] = df["Quantity"] * df["Price"]

In [53]:
df["Description"].isna().sum()

np.int64(0)

In [54]:
df["Description"] = df["Description"].fillna("UNKNOWN")

In [55]:
df.to_parquet("online_retail_cleaned.parquet", engine="pyarrow", index=False)

In [56]:
df["StockCode"].value_counts().head(30)

StockCode
85123A    5023
22423     3337
85099B    3296
84879     2692
20725     2609
21212     2557
47566     2099
20727     2045
22383     2039
21034     1950
21232     1935
22382     1935
22384     1874
21754     1852
22139     1848
20914     1822
22469     1821
20728     1820
84991     1813
POST      1803
22197     1794
82494L    1790
22386     1787
22470     1783
22138     1753
22086     1737
21931     1728
21080     1696
22411     1691
82482     1672
Name: count, dtype: int64

In [57]:
df = df[~df["StockCode"].isin(["TEST001", "TEST002"])]

In [58]:
non_product_codes = ["POST","DOT","C2","D","M","S","BANK CHARGES","AMAZONFEE","CRUK","PADS"]
df["IsProduct"] = ~df["StockCode"].isin(non_product_codes)

In [59]:
# 1. Önce kolon adlarındaki olası boşlukları standartlaştıralım
rename_map = {
    "Customer ID": "CustomerID",
    "InvoiceNo": "Invoice",
    "UnitPrice": "Price",
}
df = df.rename(
    columns={k: v for k, v in rename_map.items() if k in df.columns}
)

# 2. Customers CSV
df_customers = (
    df[["CustomerID"]]
    .rename(columns={"CustomerID": "customer_id"})
    .drop_duplicates()
)
df_customers["customer_id"] = df_customers["customer_id"].astype("Int64")   # <-- EKLE
df_customers.to_csv("customers.csv", index=False)

# 3. Products CSV
df_products = (
    df[["StockCode", "Description"]]
    .rename(columns={"StockCode": "stock_code", "Description": "description"})
    .dropna(subset=["stock_code"])
    .drop_duplicates(subset=["stock_code"], keep="last")
)
df_products.to_csv("products.csv", index=False)

# 4. Invoices CSV
df_invoices = (
    df[["Invoice", "CustomerID", "InvoiceDate", "Country"]]
    .rename(
        columns={
            "Invoice": "invoice_id",
            "CustomerID": "customer_id",
            "InvoiceDate": "invoice_date",
            "Country": "country",
        }
    )
    .drop_duplicates(subset=["invoice_id"])
)
df_invoices["customer_id"] = df_invoices["customer_id"].astype("Int64")   # <-- EKLE
df_invoices.to_csv("invoices.csv", index=False)
# 5. Invoice Items CSV
# total_amount kolonu henüz yoksa hesaplayalım
if "total_amount" not in df.columns:
    df["total_amount"] = df["Quantity"] * df["Price"]

df_invoice_items = df[
    ["Invoice", "StockCode", "Quantity", "Price", "total_amount"]
].rename(
    columns={
        "Invoice": "invoice_id",
        "StockCode": "stock_code",
        "Quantity": "quantity",
        "Price": "unit_price",
    }
)
df_invoice_items.to_csv("invoice_items.csv", index=False)

print("4 CSV dosyası da başarıyla kaydedildi!")

4 CSV dosyası da başarıyla kaydedildi!


In [60]:
pd.read_csv('products.csv').columns.tolist()

['stock_code', 'description']

In [61]:
for dosya in ["customers.csv", "products.csv", "invoices.csv", "invoice_items.csv"]:
    kolonlar = pd.read_csv(dosya, nrows=0).columns.tolist()
    print(dosya, "->", kolonlar)

customers.csv -> ['customer_id']
products.csv -> ['stock_code', 'description']
invoices.csv -> ['invoice_id', 'customer_id', 'invoice_date', 'country']
invoice_items.csv -> ['invoice_id', 'stock_code', 'quantity', 'unit_price', 'total_amount']


In [62]:
len(df)

779483

In [63]:
print(df_customers.dtypes)
print(df_customers.head(3))

customer_id    Int64
dtype: object
    customer_id
0         13085
12        13078
31        15362


In [64]:
df_invoice_items.insert(0, "id", range(1, len(df_invoice_items) + 1))
df_invoice_items.to_csv("invoice_items.csv", index=False)

In [65]:
print("customers:", len(df_customers))
print("products:", len(df_products))
print("invoices:", len(df_invoices))
print("invoice_items:", len(df_invoice_items))
print("total_amount toplamı:", df_invoice_items["total_amount"].sum())

customers: 5879
products: 4629
invoices: 36963
invoice_items: 779483
total_amount toplamı: 17374578.268


In [66]:
import os
os.makedirs("looker_data", exist_ok=True)

# 1) Aylık ciro
df["ay"] = pd.to_datetime(df["InvoiceDate"]).dt.to_period("M").astype(str)
df.groupby("ay")["total_amount"].sum().reset_index(name="net_ciro").to_csv(
    "looker_data/aylik_ciro.csv", index=False
)

# 2) Top 10 ürün
(
    df.groupby(["StockCode", "Description"])["total_amount"].sum()
    .sort_values(ascending=False).head(10)
    .reset_index(name="toplam_gelir")
    .to_csv("looker_data/top_urunler.csv", index=False)
)

# 3) Ülke dağılımı
df.groupby("Country")["total_amount"].sum().sort_values(ascending=False).reset_index(
    name="toplam_ciro"
).to_csv("looker_data/ulke_dagilimi.csv", index=False)

# 4) İptal oranı
tum_faturalar = pd.concat([
    df[["Invoice", "InvoiceDate"]].assign(iptal=False),
    df_cancelled[["Invoice", "InvoiceDate"]].assign(iptal=True),
])
tum_faturalar["ay"] = pd.to_datetime(tum_faturalar["InvoiceDate"]).dt.to_period("M").astype(str)
(
    tum_faturalar.drop_duplicates("Invoice")
    .groupby("ay")["iptal"].mean().mul(100).round(2)
    .reset_index(name="iptal_orani_yuzde")
    .to_csv("looker_data/iptal_orani.csv", index=False)
)

print("4 CSV de looker_data/ klasörüne kaydedildi")

4 CSV de looker_data/ klasörüne kaydedildi


In [67]:
non_product_codes = ["POST", "DOT", "C2", "D", "M", "S", "BANK CHARGES",
                      "AMAZONFEE", "CRUK", "PADS", "TEST001", "TEST002"]

(
    df[~df["StockCode"].isin(non_product_codes)]
    .groupby(["StockCode", "Description"])["total_amount"].sum()
    .sort_values(ascending=False).head(10)
    .reset_index(name="toplam_gelir")
    .to_csv("looker_data/top_urunler.csv", index=False)
)

In [68]:
   pd.read_csv("looker_data/top_urunler.csv")

,StockCode,Description,toplam_gelir
0,22423,REGENCY CAKESTAND 3 TIER,277656.25
1,85123A,WHITE HANGING HEART T-LIGHT HOLDER,247048.01
2,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60
3,85099B,JUMBO BAG RED RETROSPOT,134307.44
4,84879,ASSORTED COLOUR BIRD ORNAMENT,124351.86
5,47566,PARTY BUNTING,103283.38
6,23166,MEDIUM CERAMIC TOP STORAGE JAR,81416.73
7,22086,PAPER CHAIN KIT 50'S CHRISTMAS,76598.18
8,79321,CHILLI LIGHTS,69084.30
9,85099F,JUMBO BAG STRAWBERRY,64127.77
